In [1]:
# Cell 1: Setup & Constants
# Notebook 01: Dim_Client — Gold_SalesOps_Dim_Client
# Source: Party (Silver) + PartyDetails (Silver)
# Grain: PartyId (one row per source-level party record)

from pyspark.sql import functions as F

spark.conf.set("spark.sql.parquet.datetimeRebaseModeInWrite", "CORRECTED")

SILVER_BASE = "abfss://CRBDATA_PAS_EM20_PRE@onelake.dfs.fabric.microsoft.com/PAS_U_EM20_LH_Silver.Lakehouse/Tables/dbo"

print("Setup complete.")
print(f"Silver base: {SILVER_BASE}")

StatementMeta(, 04b8e3fd-5d58-4640-a60e-94b868d685f5, 3, Finished, Available, Finished, False)

Setup complete.
Silver base: abfss://CRBDATA_PAS_EM20_PRE@onelake.dfs.fabric.microsoft.com/PAS_U_EM20_LH_Silver.Lakehouse/Tables/dbo


In [2]:
# Cell 2: Load Party table

df_party = (
    spark.read.format("delta").load(f"{SILVER_BASE}/Party")
    .filter(F.col("IsDeleted") == False)
    .select(
        "PartyId",
        "GlobalPartyId",
        "SourceId",
        F.col("Party").alias("ClientName"),
        "IsActive",
        "IsIndividual",
    )
)

total_party = df_party.count()
distinct_pid = df_party.select("PartyId").distinct().count()
distinct_gpid = df_party.select("GlobalPartyId").distinct().count()

print(f"Party table (IsDeleted=False):")
print(f"  Total rows:              {total_party:,}")
print(f"  Distinct PartyId:        {distinct_pid:,}")
print(f"  Distinct GlobalPartyId:  {distinct_gpid:,}")
print(f"  Is PartyId unique?       {'YES' if distinct_pid == total_party else 'NO'}")
print(f"\nNULL counts:")
for col_name in df_party.columns:
    null_count = df_party.filter(F.col(col_name).isNull()).count()
    pct = null_count / total_party * 100 if total_party > 0 else 0
    print(f"  {col_name:<20} {null_count:>10,}  ({pct:5.1f}%)")

display(df_party.limit(10))

StatementMeta(, 04b8e3fd-5d58-4640-a60e-94b868d685f5, 4, Finished, Available, Finished, False)

Party table (IsDeleted=False):
  Total rows:              3,379,203
  Distinct PartyId:        3,379,203
  Distinct GlobalPartyId:  625,365
  Is PartyId unique?       YES

NULL counts:
  PartyId                       0  (  0.0%)
  GlobalPartyId         2,611,330  ( 77.3%)
  SourceId                      0  (  0.0%)
  ClientName                  199  (  0.0%)
  IsActive              2,192,685  ( 64.9%)
  IsIndividual            152,629  (  4.5%)


SynapseWidget(Synapse.DataFrame, bbcedcfa-2342-4714-9dda-57c978f3dc99)

In [3]:
# Cell 3: Load PartyDetails table

df_details = (
    spark.read.format("delta").load(f"{SILVER_BASE}/PartyDetails")
    .select(
        "GlobalPartyId",
        "DUNSNumber",
        "GUOPartyId",
        "Segmentation",
        "IndustryName",
        "IndustryGroupName",
        "IndustryMajorGroupName",
        "IndustryDivisionName",
        "TotalEmployeeCount",
        F.col("OperatingRevenue").alias("OperatingRevenueUSD"),
        "OwnershipType",
        "CountryName",
        "CountryCode",
        F.col("PrimaryPostalAddressCity").alias("City"),
        F.col("PrimaryPostalAddressState").alias("State"),
        "PartyType",
    )
)

total_details = df_details.count()
distinct_gpid_details = df_details.select("GlobalPartyId").distinct().count()

print(f"PartyDetails:")
print(f"  Total rows:              {total_details:,}")
print(f"  Distinct GlobalPartyId:  {distinct_gpid_details:,}")
print(f"  Is GlobalPartyId unique? {'YES' if distinct_gpid_details == total_details else 'NO'}")

display(df_details.limit(10))

StatementMeta(, 04b8e3fd-5d58-4640-a60e-94b868d685f5, 5, Finished, Available, Finished, False)

PartyDetails:
  Total rows:              2,202,177
  Distinct GlobalPartyId:  2,202,177
  Is GlobalPartyId unique? YES


SynapseWidget(Synapse.DataFrame, 86eff8dd-870b-454b-b3e3-8e194da70bdf)

In [4]:
# Cell 4: Join Party + PartyDetails on GlobalPartyId

count_before = df_party.count()

dim_client = (
    df_party
    .join(df_details, on="GlobalPartyId", how="left")
)

count_after = dim_client.count()
matched = dim_client.filter(F.col("DUNSNumber").isNotNull() | F.col("Segmentation").isNotNull()).count()
unmatched = count_after - matched

print(f"Row count before join:  {count_before:,}")
print(f"Row count after join:   {count_after:,}")
print(f"DUPLICATE CHECK:        {'PASS — no duplicates' if count_before == count_after else 'FAIL — DUPLICATES CREATED'}")
print(f"\nPartyDetails matched:   {matched:,} ({matched/count_after*100:.1f}%)")
print(f"PartyDetails unmatched: {unmatched:,} ({unmatched/count_after*100:.1f}%)")

display(dim_client.limit(10))

StatementMeta(, 04b8e3fd-5d58-4640-a60e-94b868d685f5, 6, Finished, Available, Finished, False)

Row count before join:  3,379,203
Row count after join:   3,379,203
DUPLICATE CHECK:        PASS — no duplicates

PartyDetails matched:   628,860 (18.6%)
PartyDetails unmatched: 2,750,343 (81.4%)


SynapseWidget(Synapse.DataFrame, 619b4598-9e40-4863-852e-76782e7b18f4)

In [5]:
# Cell 5: NULL Analysis for All Columns

total = dim_client.count()
print(f"Dim_Client rows: {total:,}\n")
print("Column NULL counts:")
print("-" * 55)
for col_name in dim_client.columns:
    null_count = dim_client.filter(F.col(col_name).isNull()).count()
    pct = null_count / total * 100 if total > 0 else 0
    print(f"  {col_name:<30} {null_count:>10,}  ({pct:5.1f}%)")

StatementMeta(, 04b8e3fd-5d58-4640-a60e-94b868d685f5, 7, Finished, Available, Finished, False)

Dim_Client rows: 3,379,203

Column NULL counts:
-------------------------------------------------------
  GlobalPartyId                   2,611,330  ( 77.3%)
  PartyId                                 0  (  0.0%)
  SourceId                                0  (  0.0%)
  ClientName                            199  (  0.0%)
  IsActive                        2,192,685  ( 64.9%)
  IsIndividual                      152,629  (  4.5%)
  DUNSNumber                      2,776,317  ( 82.2%)
  GUOPartyId                      3,081,693  ( 91.2%)
  Segmentation                    2,750,698  ( 81.4%)
  IndustryName                    2,807,410  ( 83.1%)
  IndustryGroupName               2,807,410  ( 83.1%)
  IndustryMajorGroupName          2,807,410  ( 83.1%)
  IndustryDivisionName            2,807,410  ( 83.1%)
  TotalEmployeeCount              2,923,014  ( 86.5%)
  OperatingRevenueUSD             2,851,397  ( 84.4%)
  OwnershipType                   2,943,949  ( 87.1%)
  CountryName                   

In [6]:
# Cell 6: GUO Name Self-Join

guo_lookup = (
    df_details
    .select(
        F.col("GlobalPartyId").alias("_guo_id"),
        F.col("CountryName").alias("_guo_dummy"),  # just to confirm join works
    )
    .join(
        spark.read.format("delta").load(f"{SILVER_BASE}/PartyDetails")
        .select(F.col("GlobalPartyId"), F.col("PartyName").alias("GUOClientName")),
        F.col("_guo_id") == F.col("GlobalPartyId"),
        how="inner"
    )
    .select(F.col("_guo_id"), "GUOClientName")
)

# Simpler approach: just lookup PartyName from PartyDetails by GUOPartyId
guo_lookup = (
    spark.read.format("delta").load(f"{SILVER_BASE}/PartyDetails")
    .select(
        F.col("GlobalPartyId").alias("_guo_id"),
        F.col("PartyName").alias("GUOClientName"),
    )
)

dim_client = (
    dim_client
    .join(guo_lookup, dim_client["GUOPartyId"] == guo_lookup["_guo_id"], how="left")
    .drop("_guo_id")
)

guo_matched = dim_client.filter(F.col("GUOClientName").isNotNull()).count()
guo_null = dim_client.filter(F.col("GUOClientName").isNull()).count()
total = dim_client.count()

print(f"GUO resolved:   {guo_matched:,} ({guo_matched/total*100:.1f}%)")
print(f"GUO unmatched:  {guo_null:,} ({guo_null/total*100:.1f}%)")
print(f"\nSample clients with GUO parent:")

display(
    dim_client
    .filter(F.col("GUOClientName").isNotNull())
    .filter(F.col("GUOPartyId") != F.col("GlobalPartyId"))
    .select("PartyId", "GlobalPartyId", "ClientName", "GUOPartyId", "GUOClientName")
    .limit(10)
)

StatementMeta(, 04b8e3fd-5d58-4640-a60e-94b868d685f5, 8, Finished, Available, Finished, False)

GUO resolved:   297,510 (8.8%)
GUO unmatched:  3,081,693 (91.2%)

Sample clients with GUO parent:


SynapseWidget(Synapse.DataFrame, a2e37b9c-59fd-4eac-b0c2-42059f084b63)

In [7]:
# Cell 7: Data Quality Overview

total = dim_client.count()

print("=== SourceId Distribution (source systems) ===")
display(
    dim_client.groupBy("SourceId")
    .agg(
        F.count("*").alias("Count"),
        F.round(F.count("*") / total * 100, 1).alias("Pct"),
    )
    .orderBy("Count", ascending=False)
)

print("\n=== Segmentation Distribution ===")
display(
    dim_client.groupBy("Segmentation")
    .agg(
        F.count("*").alias("Count"),
        F.round(F.count("*") / total * 100, 1).alias("Pct"),
    )
    .orderBy("Count", ascending=False)
)

print("\n=== Top 10 Countries ===")
display(
    dim_client.groupBy("CountryName")
    .agg(
        F.count("*").alias("Count"),
        F.round(F.count("*") / total * 100, 1).alias("Pct"),
    )
    .orderBy("Count", ascending=False)
    .limit(10)
)

print("\n=== IsActive Distribution ===")
display(dim_client.groupBy("IsActive").count().orderBy("count", ascending=False))

print("\n=== PartyIds per GlobalPartyId ===")
gpid_counts = (
    dim_client.groupBy("GlobalPartyId")
    .agg(F.count("PartyId").alias("PartyIdCount"))
    .groupBy("PartyIdCount")
    .agg(F.count("*").alias("GlobalPartyIds"))
    .orderBy("PartyIdCount")
)
display(gpid_counts)

StatementMeta(, 04b8e3fd-5d58-4640-a60e-94b868d685f5, 9, Finished, Available, Finished, False)

=== SourceId Distribution (source systems) ===


SynapseWidget(Synapse.DataFrame, e013f1b2-c0c8-41ee-8517-e9ccdaebe324)


=== Segmentation Distribution ===


SynapseWidget(Synapse.DataFrame, b70aece1-ba58-4120-aef9-cba7e7fed013)

...

SynapseWidget(Synapse.DataFrame, e999ff04-85b7-4735-9de3-68d4b5b0b371)


=== IsActive Distribution ===


SynapseWidget(Synapse.DataFrame, f9c98de4-3c22-40fc-b02f-0a6df76e39ea)


=== PartyIds per GlobalPartyId ===


SynapseWidget(Synapse.DataFrame, e0229455-3509-4684-832c-1592d154dc33)

In [8]:
# Cell 8: Write to Gold Lakehouse

dim_client.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(
    "Gold_SalesOps_Dim_Client"
)

final_count = spark.read.table("Gold_SalesOps_Dim_Client").count()
print(f"Gold_SalesOps_Dim_Client written: {final_count:,} rows")
print("=== Notebook 01 complete ===")

StatementMeta(, 04b8e3fd-5d58-4640-a60e-94b868d685f5, 10, Finished, Available, Finished, False)

Gold_SalesOps_Dim_Client written: 3,379,203 rows
=== Notebook 01 complete ===
